In [2]:
import os
import pandas as pd
import numpy as np
import pathlib

In [3]:
folder_path = r"D:\Data\Durham\Broken_Files\NewDownload"
filename = "Fixed_Combined_Per_Filament_Coordinates.csv"
savefilename = "Annotated_" + filename
chunksize = 100000

In [7]:
datafile = os.path.join(folder_path, filename)
data = pd.read_csv(datafile, on_bad_lines="warn", chunksize=chunksize)
chunksProcessed = 0
for chunk in data:
	chunk["Condition_String"] = chunk["Filename"].apply(lambda x: pathlib.Path(x).parts[-2])
	chunk["Date"] = chunk["Filename"].apply(lambda x: pathlib.Path(x).parts[-3])
	chunk["Date"] = pd.to_datetime(chunk["Date"], format="%d.%m.%y")
	chunk["Group"] = chunk["Condition_String"].apply(lambda x: 1 if x.startswith("BJ ") else 2 if x.startswith("Primary fibro ") else np.nan)
	chunk["Passage"] = chunk["Condition_String"].str.extract(r"^BJ p(\d+\.\d+) ")
	chunk["UV"] = chunk["Condition_String"].apply(lambda x: True if "+uv" in x else False if "-uv" in x else np.nan)
	chunk["Stain"] = chunk["Condition_String"].apply(lambda x: "Vimectin" if "vim" in x 
									else "Actin" if "act" in x 
									else "Tubulin" if "tub" in x
									else np.nan)
	chunk.loc[chunk[chunk["Channel"] == 1].index, "Stain"] = "Actin"
	chunk["Biological_Replicate"] = chunk["Condition_String"].str.extract(r" br(\d)")
	chunk["Technical_Replicate"] = chunk["Condition_String"].str.extract(r" tr(\d)")
	chunk["Age"] = chunk["Condition_String"].str.extract(r" (\d+) ?yo")
	chunk.to_csv(os.path.join(folder_path, savefilename), mode='a', header=not os.path.exists(os.path.join(folder_path, savefilename)), index=False)


Last two cells are just to add on that missing file from first dataset

In [1]:
file_to_append = "Per_Filament_Processed_Last_File.csv"

In [9]:
cols = pd.read_csv(os.path.join(folder_path, savefilename), nrows=0).columns
to_append_df = pd.read_csv(os.path.join(folder_path, file_to_append), on_bad_lines="warn")
to_append_df = to_append_df[cols]
to_append_df.to_csv(os.path.join(folder_path, savefilename), mode='a', header=False, index=False)